In [1]:
# Packages used
import pandas_datareader.data as web
import math
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import scipy as sp
from scipy import stats
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

In [3]:
# Collection of data from French FAMA
startdate="2000-01-01"
enddate="2024-12-31";
# Fetch Fama-French 3 Factors (monthly, U.S.)
ff3 = web.DataReader("F-F_Research_Data_Factors", "famafrench",start=startdate,end=enddate) # F_F_Research_Data_Factors is for monthly returns
# ff3 is a dictionary of tables, table[0] = factor returns
factors = ff3[0]

#Collecting Ticker data from yahoo finance!
TICKER=yf.Ticker("GS"); # Random stock
stockdata=TICKER.history(start=startdate,end=enddate,interval='1d'); 
stockdata["month-year"]=stockdata.index.strftime("%Y-%m")
stockdata["month-year"]=stockdata["month-year"].astype(str)
""" 
Need to compute the monthly stock returns. 
The formula is given by 1+ R_{monthly} = \prod_{t in a month} (1+ R_t} = closing_{last date of month}/closing_{first date of the month}"""
DATES=set(factors.index)
stockdata=stockdata[["Close","month-year"]]
for j in DATES:
    k=j.to_timestamp().strftime("%Y-%m")
    newdata=stockdata[stockdata["month-year"]==k]
    val=(newdata.iloc[-1,0]/newdata.iloc[0,0])-1;
    factors.loc[j,"Stock-RF"] = (((newdata.iloc[-1,0]/newdata.iloc[0,0])-1) - (factors.loc[j,"RF"]/100))*100
# All mmy data is in percentage, so need to convert it to percentage accordingly :-(
print(factors.info())
factors.index=factors.index.astype(str)

<>:16: SyntaxWarning: invalid escape sequence '\p'
<>:16: SyntaxWarning: invalid escape sequence '\p'
C:\Users\abhis\AppData\Local\Temp\ipykernel_2012\3857182656.py:16: SyntaxWarning: invalid escape sequence '\p'
  The formula is given by 1+ R_{monthly} = \prod_{t in a month} (1+ R_t} = closing_{last date of month}/closing_{first date of the month}"""
C:\Users\abhis\AppData\Local\Temp\ipykernel_2012\3857182656.py:5: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff3 = web.DataReader("F-F_Research_Data_Factors", "famafrench",start=startdate,end=enddate) # F_F_Research_Data_Factors is for monthly returns
C:\Users\abhis\AppData\Local\Temp\ipykernel_2012\3857182656.py:5: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype 

<class 'pandas.core.frame.DataFrame'>
PeriodIndex: 300 entries, 2000-01 to 2024-12
Freq: M
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Mkt-RF    300 non-null    float64
 1   SMB       300 non-null    float64
 2   HML       300 non-null    float64
 3   RF        300 non-null    float64
 4   Stock-RF  300 non-null    float64
dtypes: float64(5)
memory usage: 22.2 KB
None


factors columns
   'Stock-RF'  : asset excess return   (R_it - R_f)
   'Mkt-RF'     : market excess return (R_mt - R_f)
   'SMB'        : size factor
   'HML'        : value factor
   'RF'         : risk-free rate
    Index should be datetime (monthly)

In [4]:
def rolling_regression_pred(df, window=60):  # 60 months = 5 years
    preds_capm = []
    preds_ff3 = []
    actuals = []
    adj_R2_capm=[];
    adj_R2_ff3=[];
    fama_better_capm=[];
    for i in range(window, len(df)-1):
        train = df.iloc[i-window:i]
        # The y variable will always be Stock-RF
        y_train = train["Stock-RF"]

        # CAPM X variable
        X_capm = sm.add_constant(train[["Mkt-RF"]])
        capm_model = sm.OLS(y_train, X_capm).fit()

        # FF3 X variable
        X_ff3 = sm.add_constant(train[["Mkt-RF", "SMB", "HML"]])
        ff3_model = sm.OLS(y_train, X_ff3).fit()

        # Predict next period
        X_capm_test = sm.add_constant(test[["Mkt-RF"]],has_constant='add')
        X_ff3_test = sm.add_constant(test[["Mkt-RF", "SMB", "HML"]],has_constant='add')

        pred_capm = capm_model.predict(X_capm_test)
        pred_ff3 = ff3_model.predict(X_ff3_test)

        preds_capm.append(pred_capm.values[0])
        preds_ff3.append(pred_ff3.values[0])
        actuals.append(test.iloc[0,4]) # The actual value is Stock - RF 

 
    # Results DataFrame
    results = pd.DataFrame({
        "Actual": actuals,
        "CAPM_Pred": preds_capm,
        "FF3_Pred": preds_ff3,
    }, index=df.index[window+1:])
    # Compute errors
    results["CAPM_Error"] = (results["Actual"] - results["CAPM_Pred"])**2
    results["FF3_Error"] = (results["Actual"] - results["FF3_Pred"])**2
    mse_capm = np.mean(results["CAPM_Error"])
    mse_ff3 = np.mean(results["FF3_Error"])

    print("Mean Squared Error:")
    print("CAPM:", mse_capm)
    print("FF3 :", mse_ff3)

    return results

In [5]:
# Table for collecting factors, adjusted R^2 
def rolling_regression_factors(df, window=60):  # 60 months = 5 years
    column_names=list(df.columns);
    column_names=['const']+column_names;
    print(column_names);
    dates = pd.date_range(startdate, periods=25*12, freq='M')
    datelist=[dates[i].strftime("%m-%y")+" to "+dates[i+window-1].strftime("%m-%y") for i in range(0,len(dates)-59)]
    df1 = pd.DataFrame(columns=column_names[0:2]+["ADJ_R^2"], index=datelist)
    df1.columns=pd.MultiIndex.from_product([['CAPM'], df1.columns])
    df2 = pd.DataFrame(columns=column_names[0:4]+["ADJ_R^2"], index=datelist)
    df2.columns=pd.MultiIndex.from_product([['FAMA-3-FRENCH'], df2.columns])
    results=pd.concat([df1,df2],axis=1);
    for i in range(window, len(df)+1):
        train = df.iloc[i-window:i]
        y_train = train["Stock-RF"]
        # CAPM X vars
        X_capm = sm.add_constant(train[["Mkt-RF"]])
        capm_model = sm.OLS(y_train, X_capm).fit()
        
        # FF3 X vars
        X_ff3 = sm.add_constant(train[["Mkt-RF", "SMB", "HML"]])
        ff3_model = sm.OLS(y_train, X_ff3).fit()

        results.loc[datelist[i-window],"CAPM"]=list(capm_model.params)+[capm_model.rsquared_adj];
        results.loc[datelist[i-window],"FAMA-3-FRENCH"]=list(ff3_model.params)+[ff3_model.rsquared_adj];   
    # Results DataFrame
    return results

In [9]:
results = rolling_regression_factors(factors, window=60)
print("Total number of 5 year timeframe for CAPM vs FAMA comparison is", results.shape[0]);
results["FAMA better than CAPM - FACTOR"]= (results[("CAPM", "const")].apply(abs)>results[("FAMA-3-FRENCH", "const")].apply(abs));
print("The number of instances where FAMA is better than CAPM solely based on the value of the intercept is ", sum((results["FAMA better than CAPM - FACTOR"])))
results["FAMA better than CAPM - R^2"]= (results[("CAPM", "ADJ_R^2")]<results[("FAMA-3-FRENCH", "ADJ_R^2")]);
print("The number of instances where FAMA is better than CAPM solely based on adjusted R^2 is ",sum((results["FAMA better than CAPM - R^2"])))


['const', 'Mkt-RF', 'SMB', 'HML', 'RF', 'Stock-RF']


C:\Users\abhis\AppData\Local\Temp\ipykernel_2012\754092400.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(startdate, periods=25*12, freq='M')


Total number of 5 year timeframe for CAPM vs FAMA comparison is 241
The number of instances where FAMA is better than CAPM solely based on the value of the intercept is  149
The number of instances where FAMA is better than CAPM solely based on adjusted R^2 is  141


In [13]:
# See the frequency at which we are getting CAPM better than FAMA ( based on intercept value ).
counts = []
count = 0
for num in results["FAMA better than CAPM - FACTOR"]:
    if num == False:
        count += 1
    else:
        if count > 0:
            counts.append(count)
            count = 0
if count > 0:
    counts.append(count)
print(counts)
print(list(results["FAMA better than CAPM - FACTOR"]));

[2, 6, 3, 1, 3, 4, 3, 1, 9, 2, 34, 1, 12, 1, 10]
[True, False, False, True, True, False, False, False, False, False, False, True, False, False, False, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, False, False, False, True, True, True, True, False, False, False, False, True, True, True, False, False, False, True, False, True, True, False, False, False, False, False, False, False, False, False, True, True, True, False, False, True, True, True, True, True, True, True, True, True, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, False, False, False, F